In [ ]:
#imports
import pandas as pd
import numpy as np
import os
import sys

from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from collections import Counter
from sklearn import svm

In [ ]:
from google.colab import files
train_data = files.upload()
validate_data = files.upload()
test_data = files.upload()

Saving train.txt to train.txt


Saving valid.txt to valid.txt


Saving NER-test.tsv to NER-test (1).tsv


In [ ]:
#load the training data
train_features = []
train_labels = []

with open("train.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue
        if line.startswith("-DOCSTART-"):
            continue

        parts = line.split()

        word = parts[0]
        ner = parts[-1]

        train_features.append({"word": word})
        train_labels.append(ner)

In [ ]:
#load the valid data
valid_features = []
valid_labels = []

with open("valid.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue
        if line.startswith("-DOCSTART-"):
            continue

        parts = line.split()

        word = parts[0]
        ner = parts[-1]

        valid_features.append({"word": word})
        valid_labels.append(ner)

In [ ]:
#train and validate data statistics
print("Train instances:", len(train_labels))
print("Valid instances:", len(valid_labels))

train_label_distribution = Counter(train_labels)
valid_label_distribution = Counter(valid_labels)

print("\nTrain label distribution:")
print(train_label_distribution)

print("\nValid label distribution:")
print(valid_label_distribution)

Train instances: 203621
Valid instances: 51362

Train label distribution:
Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})

Valid label distribution:
Counter({'O': 42759, 'B-PER': 1842, 'B-LOC': 1837, 'B-ORG': 1341, 'I-PER': 1307, 'B-MISC': 922, 'I-ORG': 751, 'I-MISC': 346, 'I-LOC': 257})


In [ ]:
#adding all features together
training_and_valid_features = train_features + valid_features
trainingdata_length = len(train_features)

vec = DictVectorizer()

the_array = vec.fit_transform(training_and_valid_features)

#seperating the data
X_train = the_array[:trainingdata_length]
X_valid = the_array[trainingdata_length:]

In [ ]:
#prefprmance of the NERC labels
lin_clf = svm.LinearSVC()
lin_clf.fit(X_train, train_labels)

y_predictions = lin_clf.predict(X_valid)
print(classification_report(valid_labels, y_predictions))

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


              precision    recall  f1-score   support

       B-LOC       0.87      0.80      0.83      1837
      B-MISC       0.87      0.73      0.79       922
       B-ORG       0.79      0.65      0.71      1341
       B-PER       0.85      0.62      0.72      1842
       I-LOC       0.72      0.70      0.71       257
      I-MISC       0.79      0.50      0.61       346
       I-ORG       0.71      0.45      0.55       751
       I-PER       0.70      0.41      0.52      1307
           O       0.95      1.00      0.98     42759

    accuracy                           0.94     51362
   macro avg       0.81      0.65      0.71     51362
weighted avg       0.93      0.94      0.93     51362



In [ ]:
#training data
test_path = "NER-test.tsv"

#debugging
test_df = pd.read_csv(
    test_path,
    sep="\t",
    engine="python",
    on_bad_lines="warn"
)

test_features = []

for word in test_df["token"]:
    test_features.append({"word": word})

/tmp/ipykernel_3839/2261907370.py:5: ParserWarning: Skipping line 33: Expected 4 fields in line 33, saw 5

  test_df = pd.read_csv(


In [ ]:
test_labels = test_df["BIO_NER_tag"]

X_test = vec.transform(test_features)
test_predictions = lin_clf.predict(X_test)

#evaluation
print(classification_report(test_labels, test_predictions))

               precision    recall  f1-score   support

        B-LOC       0.67      0.57      0.62         7
       B-MISC       0.00      0.00      0.00         0
        B-ORG       0.40      0.67      0.50         3
        B-PER       0.00      0.00      0.00         0
     B-PERSON       0.00      0.00      0.00        11
B-WORK_OF_ART       0.00      0.00      0.00         9
        I-LOC       0.00      0.00      0.00         1
        I-ORG       0.00      0.00      0.00         2
        I-PER       0.00      0.00      0.00         0
     I-PERSON       0.00      0.00      0.00         8
I-WORK_OF_ART       0.00      0.00      0.00        10
            O       0.85      0.99      0.92       185

     accuracy                           0.81       236
    macro avg       0.16      0.19      0.17       236
 weighted avg       0.69      0.81      0.74       236



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_